# Winnie Tai 遊戲室：用免費 GPU 錄聲音 C、D

1. 上方選單「執行階段 → 變更執行階段類型」，選 **T4 GPU**，按儲存。
2. 由上往下，每一格按左邊的 ▶ 執行（或「執行階段 → 全部執行」）。
3. 第 1 格會問能不能連 Google 雲端硬碟，請按允許：錄好的聲音存在雲端硬碟，中途斷線也不會不見。
4. 斷線或關掉後，重新打開這個筆記本再「全部執行」，會從上次停的地方接著錄。
5. 全部錄完，最後一格會下載 `winnie_voice_CD.zip`，把它傳回專案就好。


In [ ]:
# 1. 檢查 GPU、連接雲端硬碟
import subprocess, os
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or '⚠️ 沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU')
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/winnie_voice'
os.makedirs(OUT, exist_ok=True)
VOICES = ['C', 'D']   # 要錄的聲音


In [ ]:
# 2. 安裝 BreezyVoice（約 3–5 分鐘）
%cd /content
!test -d BreezyVoice || git clone --depth 1 https://github.com/mtkresearch/BreezyVoice.git
!test -d winnie-playroom || git clone --depth 1 https://github.com/gelu27/winnie-playroom.git
!git -C winnie-playroom pull -q
!pip install -q conformer diffusers hydra-core HyperPyYAML inflect librosa lightning omegaconf openai-whisper soundfile WeTextProcessing opencc-python-reimplemented onnxruntime "ruamel.yaml<0.18" "transformers>=4.40,<5" gdown wget matplotlib networkx rich
!pip install -q --no-deps g2pw
print('安裝完成')


In [ ]:
# 3. 開始錄音（C、D 各約 600 句；中斷後重跑這格會接著錄）
import os, subprocess
T = '/content/winnie-playroom/tools/colab'
total = sum(1 for l in open(f'{T}/lines.tsv', encoding='utf-8') if l.strip())
for v in VOICES:
    out = f'{OUT}/out_{v}'; os.makedirs(out, exist_ok=True)
    print(f'聲音 {v}：已經有 {len(os.listdir(out))}／{total} 句，繼續錄…', flush=True)
    ref_text = open(f'{T}/ref/{v}.txt', encoding='utf-8').read()
    p = subprocess.Popen(['python', f'{T}/breezy_run.py', f'{T}/ref/{v}.wav', ref_text, out, f'{T}/lines.tsv'],
                         cwd='/content/BreezyVoice', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, 'PYTHONUTF8': '1'})
    n = len(os.listdir(out))
    for line in p.stdout:
        if line.startswith('DONE'):
            n += 1
            if n % 20 == 0: print(f'  聲音 {v}：{n}／{total}', flush=True)
        elif 'Error' in line or 'Traceback' in line:
            print(line, end='')
    p.wait()
    print(f'聲音 {v} 完成：{len(os.listdir(out))}／{total}')


In [ ]:
# 4. 轉成 mp3 並打包下載（錄完再執行）
import shutil, subprocess, os
from google.colab import files
pack = '/content/winnie_voice_CD'
shutil.rmtree(pack, ignore_errors=True)
for v in VOICES:
    src, dst = f'{OUT}/out_{v}', f'{pack}/{v}'
    os.makedirs(dst, exist_ok=True)
    for f in sorted(os.listdir(src)):
        if f.endswith('.wav'):
            subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f'{src}/{f}', '-b:a', '128k', f'{dst}/{f[:-4]}.mp3'])
    print(v, len(os.listdir(dst)), '句')
z = shutil.make_archive(pack, 'zip', pack, '.')
print('zip 大小約', round(os.path.getsize(z) / 1e6), 'MB')
files.download(z)
